In [3]:
import os
import psycopg2
import pandas as pd
import warnings

warnings.filterwarnings("ignore")


file_path = "data.csv"  
ecom_df = pd.read_csv(file_path)


ecom_df = ecom_df.loc[:, ~ecom_df.columns.str.contains('^Unnamed')]


ecom_df.columns = ecom_df.columns.str.strip()
ecom_df.columns = [col.replace(" ", "_").replace("(", "").replace(")", "").replace("%", "pct") 
                   for col in ecom_df.columns]


def infer_sql_type(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return "INT"
    elif pd.api.types.is_float_dtype(dtype):
        return "FLOAT"
    elif pd.api.types.is_bool_dtype(dtype):
        return "BOOLEAN"
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return "TIMESTAMP"
    else:
        return "TEXT"


conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="root", 
    host="localhost",
    port="5432"
)
cur = conn.cursor()


table_name = "ecommerce_data"
columns = ecom_df.dtypes
sql_columns = ",\n  ".join([f'"{col}" {infer_sql_type(dtype)}' for col, dtype in columns.items()])

create_stmt = f"""
CREATE TABLE "{table_name}" (
  {sql_columns}
);
"""

cur.execute(f'DROP TABLE IF EXISTS "{table_name}" CASCADE;')
cur.execute(create_stmt)
conn.commit()
print("Table recreated with schema:")
print(create_stmt)


columns_list = list(ecom_df.columns)
placeholders = ', '.join(['%s'] * len(columns_list))
quoted_cols = ', '.join([f'"{col}"' for col in columns_list])

insert_stmt = f'INSERT INTO "{table_name}" ({quoted_cols}) VALUES ({placeholders})'

for _, row in ecom_df.iterrows():
    row_values = [None if pd.isna(val) else val for val in row[columns_list]]
    cur.execute(insert_stmt, tuple(row_values))

conn.commit()
print("Data inserted successfully")


df_from_db = pd.read_sql(f'SELECT * FROM "{table_name}" LIMIT 5', conn)
print("Data fetched from DB:")
print(df_from_db.head())

cur.close()
conn.close()


Table recreated with schema:

CREATE TABLE "ecommerce_data" (
  "Order_ID" TEXT,
  "Customer_ID" TEXT,
  "Platform" TEXT,
  "Order_Date_&_Time" TEXT,
  "Delivery_Time_Minutes" INT,
  "Product_Category" TEXT,
  "Order_Value_INR" INT,
  "Customer_Feedback" TEXT,
  "Service_Rating" INT,
  "Delivery_Delay" TEXT,
  "Refund_Requested" TEXT
);

Data inserted successfully
Data fetched from DB:
    Order_ID Customer_ID Platform Order_Date_&_Time  Delivery_Time_Minutes  \
0  ORD000001    CUST2824  JioMart           19:29.5                     30   
1  ORD000002    CUST1409  Blinkit           54:29.5                     16   
2  ORD000003    CUST5506  JioMart           21:29.5                     25   
3  ORD000004    CUST5012  JioMart           19:29.5                     42   
4  ORD000005    CUST4657  Blinkit           49:29.5                     30   

      Product_Category  Order_Value_INR              Customer_Feedback  \
0  Fruits & Vegetables              382  Fast delivery, great servic